# Day 3 — Feature Engineering + Preprocessing

**Goal:** cleaned table → engineered scores → encoded/scaled **X** and **y**.

> **Read first:** [CONCEPTS.md](./CONCEPTS.md)


## Setup paths and imports

Run with cwd `notebooks/day03/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "dataset").is_dir() and (p / "requirements.txt").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found. cwd should be notebooks/dayXX/.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    add_engineered_features,
    load_cleaned_table,
    prepare_train_test_bundle,
    save_artifacts,
    ENGINEERED_COLS,
)

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")

df = load_cleaned_table(PROJECT_ROOT)
df.shape

## 1. Feature engineering

In [ ]:
engineered = add_engineered_features(df)
engineered[ENGINEERED_COLS + ["Career"]].head()

In [ ]:
engineered[ENGINEERED_COLS].describe().round(2)

## 2. Train/test split + preprocessor (fit on train only)

In [ ]:
bundle = prepare_train_test_bundle(engineered, test_size=0.2, random_state=42)
bundle["n_train"], bundle["n_test"], bundle["n_features"]

In [ ]:
print("Career classes:", len(bundle["career_classes"]))
bundle["feature_names"][:6], "...", bundle["feature_names"][-2:]

## 3. Save pipeline for Day 4+

In [ ]:
paths = save_artifacts(bundle, PROJECT_ROOT)
paths

## 4. Stratified split check

In [ ]:
import pandas as pd

train_counts = pd.Series(bundle["y_train"]).value_counts().sort_index()
test_counts = pd.Series(bundle["y_test"]).value_counts().sort_index()
compare = pd.DataFrame({"train": train_counts, "test": test_counts})
compare.index = [bundle["career_classes"][i] for i in compare.index]
compare.head(8)